In [4]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
from sqlalchemy import text
import numpy as np
import oracledb
import sys

#CONEXIONES DESTINO

DB_USER=config("USER2_BDI_DW")
DB_PASSWORD=config("PASS2_BDI_DW")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_DW")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb

un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname = config("HOST4_BDI_POSTGRES")
service_name = "WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@', connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

# Cargar datos de referencia desde PostgreSQL
tipdoc = pd.read_sql_query("SELECT id_tipdoc, cod_tdo FROM dim_tipdoc", con=engine2)
sexo = pd.read_sql_query("SELECT id_sexo, cod_sex FROM dim_sexo", con=engine2)
estciv = pd.read_sql_query("SELECT id_estciv, cod_ecv FROM dim_estciv", con=engine2)
tipseg = pd.read_sql_query("SELECT id_tipseg, cod_tse FROM dim_tipseg", con=engine2)
plansalud = pd.read_sql_query("SELECT id_plansalud, cod_tse, cod_psa FROM dim_plansalud", con=engine2)
cas = pd.read_sql_query("SELECT id_cas, cod_cas, cod_red FROM dim_cas WHERE cod_cas IS NOT NULL", con=connection2)
tipoparen = pd.read_sql_query("SELECT id_tippa, cod_tippa FROM dim_tippa", con=engine2)
estreg = pd.read_sql_query("SELECT id_estreg, cod_est FROM dim_estreg", con=engine2)

# Función para limpiar columnas
def strip_columns(df):
    return df.apply(lambda x: x.str.strip() if x.dtype == "object" and x.apply(type).eq(str).all() else x)

tipdoc = strip_columns(tipdoc)
sexo = strip_columns(sexo)
estciv = strip_columns(estciv)
tipseg = strip_columns(tipseg)
plansalud = strip_columns(plansalud)
cas = strip_columns(cas)
tipoparen = strip_columns(tipoparen)
estreg = strip_columns(estreg)

# Obtener y ajustar la fecha de inicio
fec_ini = pd.read_sql_query("SELECT fec_ini FROM etl_act WHERE id_proceso=1", con=connection2)
fec_ini = fec_ini.iloc[0, 0] - timedelta(days=1)  # Restar 1 día a la fecha de inicio

# Consulta para obtener toda la data relevante desde Oracle
query = f"""
SELECT 
    p.PERSECNUM,
    p.PERTIPDOCIDENCOD,
    p.PERDOCIDENNUM,
    p.PERPRINOMDES,
    p.PERSEGNOMDES,
    p.PERAPEPATDES,
    p.PERAPEMATDES,
    p.PERNACFEC,
    p.PERSEXOCOD,
    p.PERESTCIVCOD,
    p.PERTIPSEGCOD,
    p.PERPLANSALUCOD,
    p.PERCENASIADSCOD,
    p.PERINSFEC,
    p.PERVIGFEC,
    p.PERFALFEC,
    p.PERCREFEC,
    p.PERMODFEC,
    p.PERTIPOPARECOD,
    p.PERUBIGEONAC,
    p.PERUBIGEODOM,
    p.PERESTREGCOD
FROM SGSS.CMPER10 p
WHERE p.PERCREFEC >= TO_DATE('{fec_ini.strftime('%Y-%m-%d')}', 'YYYY-MM-DD') 
    OR p.PERMODFEC >= TO_DATE('{fec_ini.strftime('%Y-%m-%d')}', 'YYYY-MM-DD')
"""

# Cargar todos los datos en un DataFrame
chunk = pd.read_sql_query(query, con=engine0)

chunk = chunk.rename(columns={
        'persecnum': 'per_sec',
        'pertipdocidencod': 'cod_tdo',
        'perdocidennum': 'num_doc',
        'perprinomdes': 'pri_nom',
        'persegnomdes': 'seg_nom',
        'perapepatdes': 'ape_pat',
        'perapematdes': 'ape_mat',
        'pernacfec': 'fec_nac',
        'persexocod': 'cod_sex',
        'perestcivcod': 'cod_ecv',
        'pertipsegcod': 'cod_tse',
        'perplansalucod': 'cod_psa',
        'percenasiadscod': 'cod_cas',
        'perinsfec': 'fec_ins',
        'pervigfec': 'fec_vig',
        'perfalfec': 'fec_fal',
        'percrefec': 'fec_cre',
        'permodfec': 'fec_mod',
        'pertipoparecod': 'cod_tippa',
        'perubigeonac': 'ubi_nac',
        'perubigeodom': 'ubi_dom',
        'perestregcod': 'cod_est'
    })
    
chunk = strip_columns(chunk)

chunk = pd.merge(chunk, tipdoc, on='cod_tdo', how='left').drop('cod_tdo', axis=1)

chunk = pd.merge(chunk, sexo, on='cod_sex', how="left").drop('cod_sex', axis=1)

chunk = pd.merge(chunk, estciv, on='cod_ecv', how='left').drop('cod_ecv', axis=1)

chunk = pd.merge(chunk, plansalud, on=['cod_tse', 'cod_psa'], how="left").drop('cod_psa', axis=1)

chunk = pd.merge(chunk, tipseg, on='cod_tse', how="left").drop('cod_tse', axis=1)

chunk = pd.merge(chunk, cas,how='left',on='cod_cas').drop('cod_cas', axis=1)

chunk = pd.merge(chunk, tipoparen, on='cod_tippa', how="left").drop('cod_tippa', axis=1)

chunk = pd.merge(chunk, estreg, on='cod_est', how="left").drop('cod_est', axis=1)

# Convertir columnas de fecha a datetime y manejar NaT
date_columns = ['fec_nac', 'fec_ins', 'fec_vig', 'fec_fal', 'fec_cre', 'fec_mod']

for col in date_columns:
    chunk[col] = pd.to_datetime(chunk[col], errors='coerce')
for col in date_columns:
    chunk[col] = chunk[col].apply(lambda x: None if pd.isna(x) else x)

# Convertir el resto de NaN a None
chunk = chunk.applymap(lambda x: None if pd.isna(x) else x)

# Upsert en PostgreSQL
for _, row in chunk.iterrows():
    row_dict = row.to_dict()

    for key in date_columns:
        if row_dict[key] == 'NaT' or pd.isna(row_dict[key]):
            row_dict[key] = None
            
    def clean_data(row_dict):
        for key, value in row_dict.items():
            if isinstance(value, str):
                row_dict[key] = value.replace('\x00', '')
        return row_dict

    row_dict = clean_data(row_dict)

    # Verificar si el registro ya existe
    check_query = text("""
    SELECT 1 FROM dim_paciente WHERE per_sec = :per_sec
    """)
    result = engine2.execute(check_query, per_sec=row_dict['per_sec']).fetchone()

    if result:
        # Si existe, actualizar el registro
        update_query = text("""
        UPDATE dim_paciente
        SET id_tipdoc=:id_tipdoc, num_doc=:num_doc, pri_nom=:pri_nom, seg_nom=:seg_nom, ape_pat=:ape_pat, ape_mat=:ape_mat, 
            fec_nac=:fec_nac, id_sexo=:id_sexo, id_estciv=:id_estciv, id_tipseg=:id_tipseg, id_cas=:id_cas, fec_ins=:fec_ins, 
            fec_vig=:fec_vig, fec_fal=:fec_fal, fec_cre=:fec_cre, fec_mod=:fec_mod, id_plansalud=:id_plansalud, 
            id_tippa=:id_tippa, ubi_nac=:ubi_nac, ubi_dom=:ubi_dom, id_estreg=:id_estreg
        WHERE per_sec = :per_sec
        """)
        engine2.execute(update_query, **row_dict)
    else:
        # Si no existe, insertar un nuevo registro
        insert_query = text("""
        INSERT INTO dim_paciente (per_sec, id_tipdoc, num_doc, pri_nom, seg_nom, ape_pat, ape_mat, fec_nac, id_sexo, 
                                id_estciv, id_tipseg, id_cas, fec_ins, fec_vig, fec_fal, fec_cre, fec_mod, 
                                id_plansalud, id_tippa, ubi_nac, ubi_dom, id_estreg)
        VALUES (:per_sec, :id_tipdoc, :num_doc, :pri_nom, :seg_nom, :ape_pat, :ape_mat, :fec_nac, :id_sexo, 
                :id_estciv, :id_tipseg, :id_cas, :fec_ins, :fec_vig, :fec_fal, :fec_cre, :fec_mod, 
                :id_plansalud, :id_tippa, :ubi_nac, :ubi_dom, :id_estreg)
        """)
        engine2.execute(insert_query, **row_dict)

In [5]:
now = datetime.now().strftime('%Y-%m-%d')
query1=f"UPDATE etl_act SET fec_ini ='{now}' WHERE id_proceso=1"
query2=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_proceso=1"
c1= text(query1)
c2= text(query2)
connection2.execute(c1)
connection2.execute(c2)

In [6]:
connection2.close()
connection0.close()
engine2.dispose()
engine0.dispose()